In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path(r"C:\Users\dunca\OneDrive\Documents\UON\SCS6104\Capstone")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.etl import enrich

# Test geo-only on a handful of known IPs
test_ips = [
    "205.174.165.73",  # documented CICIDS attacker
    "8.8.8.8",          # Google DNS
    "1.1.1.1",          # Cloudflare DNS
    "192.168.10.50",    # internal (should fail gracefully)
]

geo = enrich.enrich_geo(test_ips)
for ip, data in geo.items():
    print(f"\n{ip}:")
    for k, v in data.items():
        print(f"  {k}: {v}")

13:37:24 | INFO    | etl          | Logging to: C:\Users\dunca\OneDrive\Documents\UON\SCS6104\Capstone\logs\etl_20260511_133724.log
13:37:24 | INFO    | etl.enrich   | Opening MaxMind DB: GeoLite2-City.mmdb
13:37:24 | INFO    | etl.enrich   | GeoIP: 3 resolved, 1 not-found, 0 errors

205.174.165.73:
  country_iso: CA
  country_name: Canada
  city: Fredericton
  latitude: 45.8122
  longitude: -66.6784
  asn: None
  asn_org: None

8.8.8.8:
  country_iso: US
  country_name: United States
  city: None
  latitude: 37.751
  longitude: -97.822
  asn: None
  asn_org: None

1.1.1.1:
  country_iso: None
  country_name: None
  city: None
  latitude: None
  longitude: None
  asn: None
  asn_org: None

192.168.10.50:
  country_iso: None
  country_name: None
  city: None
  latitude: None
  longitude: None
  asn: None
  asn_org: None


In [2]:
import os
from dotenv import load_dotenv

load_dotenv(PROJECT_ROOT / ".env")
api_key = os.getenv("ABUSEIPDB_API_KEY")

# Test with just 1 IP — the documented attacker
test_ip = ["205.174.165.73"]
rep = enrich.enrich_reputation(test_ip, api_key)
print(rep)

13:38:57 | INFO    | etl.enrich   | Querying AbuseIPDB for 1 IPs (throttled to ~10 req/s)
13:38:58 | INFO    | etl.enrich   | AbuseIPDB: 1 IPs enriched (complete)
{'205.174.165.73': {'abuse_confidence': 0, 'is_known_attacker': 0}}


In [3]:
from src.etl.context import PipelineContext
from src.etl.run_tracker import PipelineRun
from src.etl import extract, transform

ctx = PipelineContext(run_mode="sample", sample_size=10_000)

with PipelineRun(ctx) as run:
    with run.stage("extract") as s:
        extract.run(ctx)
        s.set_metrics(rows_in=0, rows_out=ctx.rows_extracted)
    with run.stage("transform") as s:
        transform.run(ctx)
        s.set_metrics(rows_in=ctx.rows_extracted, rows_out=ctx.rows_transformed)
    with run.stage("enrich") as s:
        enriched = enrich.run(ctx)
        s.set_metrics(rows_in=ctx.rows_transformed, rows_out=ctx.rows_enriched)

print(f"\n✅ Enriched {ctx.rows_enriched} unique IPs")

# Show a few enriched records
import pandas as pd
enriched_df = pd.DataFrame.from_dict(enriched, orient="index").reset_index().rename(columns={"index": "ip"})
print("\nSample enriched records:")
print(enriched_df.head(10).to_string(index=False))

13:39:49 | INFO    | etl.tracker  | ╔══ Pipeline run 11 started: cicids_to_warehouse (sample) ══╗
13:39:49 | INFO    | etl.tracker  | ▶ Starting stage: extract
13:39:49 | INFO    | etl.extract  | Reading stratified sample (n=10,000) from cicids_clean.parquet


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

13:40:12 | INFO    | etl.extract  | Loaded 8,047 rows × 87 columns into memory
13:40:12 | WARNING | etl.extract  | Dropped 1,000 invalid rows (missing timestamp or IPs)
13:40:12 | INFO    | etl.extract  | Attack family distribution (top): {'Benign': 1000, 'Reconnaissance': 1000, 'DDoS': 1000, 'DoS': 1000, 'Botnet': 1000, 'Web Attack': 1000, 'Brute Force': 1000, 'Infiltration': 36, 'Exploit': 11}
13:40:12 | INFO    | etl.tracker  | ✓ Stage 'extract' completed in 23.4s (rows: 0 → 7,047)
13:40:12 | INFO    | etl.tracker  | ▶ Starting stage: transform
13:40:12 | INFO    | etl.transform | Transforming 7,047 rows
13:40:12 | INFO    | etl.transform | Deriving date_sk, time_sk, is_after_hours, is_weekend
13:40:12 | INFO    | etl.transform | Classifying source IPs
13:40:13 | INFO    | etl.transform | Classifying destination IPs
13:40:13 | INFO    | etl.transform | Source IP classification: {'internal': 6536, 'external': 511}
13:40:13 | INFO    | etl.transform | Destination IP classification: {'